In [3]:
import os
import io

import pandas as pd

import pdfplumber
from pypdf import PdfReader

from PIL import Image

# --------------------
# Paths
# --------------------
pdf_path = "../data/raw/0-آثارنا المنهوبة_3.pdf"
pdf_name = os.path.basename(pdf_path)

# --------------------
# Helper : ignore corner images
# --------------------
MAX_PAGE_RATIO = 0.6
def is_in_corner(img, page_width, page_height):
    x0, y0, x1, y1 = img['x0'], img['y0'], img['x1'], img['y1']
    if x0 > 350 and y0 > 750: return True
    if x1 < 250 and y0 > 750: return True
    if x1 < 250 and y1 < 50:  return True
    if x0 > 350 and y1 < 50:  return True
    return False

# --------------------
# INIT
# --------------------
chunks = []
chunks_images = []

# --------------------
# LECTURE PDF
# --------------------
with pdfplumber.open(pdf_path) as pdf:
    reader = PdfReader(pdf_path)
    
    for page_number, (pdfpl_page, pypdf_page) in enumerate(zip(pdf.pages, reader.pages), start=1):
        page_width = pdfpl_page.width
        page_height = pdfpl_page.height

        # =========================
        # 1️⃣ TABLES → CHUNKS
        # =========================
        tables = pdfpl_page.find_tables()
 
        for table_idx, t in enumerate(tables, start=1):
            x0, top, x1, bottom = t.bbox
            centre_x = (x0 + x1)/2
            centre_y = (top + bottom)/2

            chunk = {
                "pdf_name": pdf_name,
                "page_number": page_number,
                "chunk_id": f"{pdf_name}_page{page_number}_table{table_idx}",
                "table_idx": table_idx,
                "bbox": (x0, top, x1, bottom),
                "centre": (centre_x, centre_y)
            }
            chunks.append(chunk)

        # =========================
        # 2️⃣ IMAGES → EXTRACTION + TABLE
        # =========================
        images = pdfpl_page.images

        for i, img in enumerate(images, start=1):
            width = img['x1'] - img['x0']
            height = img['y1'] - img['y0']

            if is_in_corner(img, page_width, page_height):
                continue
            if width > page_width * MAX_PAGE_RATIO or height > page_height * MAX_PAGE_RATIO:
                continue

            x_centre_img = (img['x0'] + img['x1']) / 2
            y_centre_img = (img['y0'] + img['y1']) / 2

            # --- extraction image PNG ---
            try:
                img_key = list(pypdf_page.images.keys())[i-1]
                img_obj = pypdf_page.images[img_key]
                im = Image.open(io.BytesIO(img_obj.data))
            except (IndexError, OSError):
                continue

            img_filename = f"{pdf_name.replace('.pdf','')}_page{page_number}_img{i}.png"
            img_path = os.path.join(output_dir_img, img_filename)
            im.save(img_path)

            # --- trouver chunk le plus proche ---
            chunk_id = None
            table_idx_match = None

            chunks_in_page = [c for c in chunks if c['page_number'] == page_number]
            if chunks_in_page:
                closest_chunk = min(
                    chunks_in_page,
                    key=lambda c: ((x_centre_img - c['centre'][0])**2 + (y_centre_img - c['centre'][1])**2)
                )
                chunk_id = closest_chunk["chunk_id"]
                table_idx_match = closest_chunk["table_idx"]

            # --- ajouter à la table images ---
            chunks_images.append({
                "pdf_name": pdf_name,
                "page_number": page_number,
                "img_filename": img_filename,
                "centre": (x_centre_img, y_centre_img),
                "chunk_id": chunk_id,
                "table_idx": table_idx_match
            })

# =========================
# 3️⃣ DATAFRAMES
# =========================

df_tables = pd.DataFrame(chunks)

df_images = pd.DataFrame(chunks_images)

# =========================
# 4️⃣ EXPORT
# =========================

df_tables.to_csv("../data/processed/pdf_tables.csv", index=False, encoding="utf-8-sig")
df_images.to_csv("../data/processed/pdf_images.csv", index=False, encoding="utf-8-sig")

print(df_tables.head())
print(df_images.head())

NameError: name 'output_dir_img' is not defined

In [4]:
import os
import io

import pandas as pd
import pdfplumber
from pypdf import PdfReader
from PIL import Image

# --------------------
# Paths
# --------------------
pdf_path = "../data/raw/0-آثارنا المنهوبة_3.pdf"
pdf_name = os.path.basename(pdf_path)

output_dir_img = "../figures/pdf_images"
csv_images_path = "../data/processed/pdf_images.csv"

os.makedirs(output_dir_img, exist_ok=True)
os.makedirs(os.path.dirname(csv_images_path), exist_ok=True)

# --------------------
# Filtre images
# --------------------
MAX_PAGE_RATIO = 0.6

def is_in_corner(img, page_width, page_height):
    x0, y0, x1, y1 = img['x0'], img['y0'], img['x1'], img['y1']
    if x0 > 350 and y0 > 750: return True
    if x1 < 250 and y0 > 750: return True
    if x1 < 250 and y1 < 50:  return True
    if x0 > 350 and y1 < 50:  return True
    return False

# --------------------
# INIT
# --------------------
images_data = []

# --------------------
# LECTURE PDF
# --------------------
with pdfplumber.open(pdf_path) as pdf:
    reader = PdfReader(pdf_path)

    for page_number, (pdfpl_page, pypdf_page) in enumerate(zip(pdf.pages, reader.pages), start=1):
        page_width = pdfpl_page.width
        page_height = pdfpl_page.height

        images = pdfpl_page.images
        image_keys = list(pypdf_page.images.keys())

        for i, img in enumerate(images, start=1):
            width = img['x1'] - img['x0']
            height = img['y1'] - img['y0']

            # -------- filtres --------
            if is_in_corner(img, page_width, page_height):
                continue
            if width > page_width * MAX_PAGE_RATIO or height > page_height * MAX_PAGE_RATIO:
                continue

            # centres
            x_centre = (img['x0'] + img['x1']) / 2
            y_centre = (img['y0'] + img['y1']) / 2

            # -------- extraction --------
            try:
                if i-1 >= len(image_keys):
                    continue
                img_key = image_keys[i-1]
                img_obj = pypdf_page.images[img_key]
                im = Image.open(io.BytesIO(img_obj.data))
            except Exception:
                continue

            # -------- sauvegarde --------
            img_filename = f"{pdf_name.replace('.pdf','')}_page{page_number}_img{i}.png"
            img_path = os.path.join(output_dir_img, img_filename)
            im.save(img_path)

            # -------- stockage --------
            images_data.append({
                "pdf_name": pdf_name,
                "page_number": page_number,
                "img_filename": img_filename,
                "width": width,
                "height": height,
                "centre_x": x_centre,
                "centre_y": y_centre
            })

# --------------------
# DATAFRAME
# --------------------
df_images = pd.DataFrame(images_data)

# --------------------
# EXPORT
# --------------------
df_images.to_csv(csv_images_path, index=False, encoding="utf-8-sig")

print("✅ Images extraites :", len(df_images))
print(df_images.head())

✅ Images extraites : 24
                  pdf_name  page_number                        img_filename  \
0  0-آثارنا المنهوبة_3.pdf            1  0-آثارنا المنهوبة_3_page1_img3.png   
1  0-آثارنا المنهوبة_3.pdf            1  0-آثارنا المنهوبة_3_page1_img4.png   
2  0-آثارنا المنهوبة_3.pdf            1  0-آثارنا المنهوبة_3_page1_img5.png   
3  0-آثارنا المنهوبة_3.pdf            2  0-آثارنا المنهوبة_3_page2_img3.png   
4  0-آثارنا المنهوبة_3.pdf            2  0-آثارنا المنهوبة_3_page2_img4.png   

    width  height  centre_x  centre_y  
0  127.00  181.65   140.300   371.795  
1  129.00  168.75   153.750   166.695  
2  183.05  213.99   141.025   614.855  
3  234.75  170.25   170.625   607.345  
4  151.45   94.15   191.525   139.745  


In [5]:
import os
import io

import pandas as pd
import pdfplumber
from pypdf import PdfReader
from PIL import Image

# --------------------
# Paths
# --------------------
pdf_path = "../data/raw/0-آثارنا المنهوبة_3.pdf"
pdf_name = os.path.basename(pdf_path)

output_dir_img = "../figures/pdf_images"
csv_images_path = "../data/processed/pdf_images.csv"

os.makedirs(output_dir_img, exist_ok=True)
os.makedirs(os.path.dirname(csv_images_path), exist_ok=True)

# --------------------
# Filtre images
# --------------------
MAX_PAGE_RATIO = 0.6

def is_in_corner(img, page_width, page_height):
    x0, y0, x1, y1 = img['x0'], img['y0'], img['x1'], img['y1']
    if x0 > 350 and y0 > 750: return True
    if x1 < 250 and y0 > 750: return True
    if x1 < 250 and y1 < 50:  return True
    if x0 > 350 and y1 < 50:  return True
    return False

# --------------------
# INIT
# --------------------
images_data = []

# --------------------
# LECTURE PDF
# --------------------
with pdfplumber.open(pdf_path) as pdf:
    reader = PdfReader(pdf_path)

    for page_number, (pdfpl_page, pypdf_page) in enumerate(zip(pdf.pages, reader.pages), start=1):
        page_width = pdfpl_page.width
        page_height = pdfpl_page.height

        # 👉 tables LIGHT (juste centres)
        tables = pdfpl_page.find_tables()
        page_chunks = []

        for table_idx, t in enumerate(tables, start=1):
            x0, top, x1, bottom = t.bbox
            centre_x = (x0 + x1) / 2
            centre_y = (top + bottom) / 2

            page_chunks.append({
                "chunk_id": f"{pdf_name}_page{page_number}_table{table_idx}",
                "table_idx": table_idx,
                "centre": (centre_x, centre_y)
            })

        # 👉 images
        images = pdfpl_page.images
        image_keys = list(pypdf_page.images.keys())

        for i, img in enumerate(images, start=1):
            width = img['x1'] - img['x0']
            height = img['y1'] - img['y0']

            if is_in_corner(img, page_width, page_height):
                continue
            if width > page_width * MAX_PAGE_RATIO or height > page_height * MAX_PAGE_RATIO:
                continue

            x_centre = (img['x0'] + img['x1']) / 2
            y_centre = (img['y0'] + img['y1']) / 2

            # extraction
            try:
                if i-1 >= len(image_keys):
                    continue
                img_key = image_keys[i-1]
                img_obj = pypdf_page.images[img_key]
                im = Image.open(io.BytesIO(img_obj.data))
            except Exception:
                continue

            img_filename = f"{pdf_name.replace('.pdf','')}_page{page_number}_img{i}.png"
            img_path = os.path.join(output_dir_img, img_filename)
            im.save(img_path)

            # 👉 matching table la + proche
            chunk_id = None
            table_idx_match = None

            if page_chunks:
                closest = min(
                    page_chunks,
                    key=lambda c: (
                        (x_centre - c['centre'][0])**2 +
                        (y_centre - c['centre'][1])**2
                    )
                )
                chunk_id = closest["chunk_id"]
                table_idx_match = closest["table_idx"]

            images_data.append({
                "pdf_name": pdf_name,
                "page_number": page_number,
                "img_filename": img_filename,
                "chunk_id": chunk_id,
                "table_idx": table_idx_match,
                "centre_x": x_centre,
                "centre_y": y_centre
            })

# --------------------
# DF + EXPORT
# --------------------
df_images = pd.DataFrame(images_data)
df_images.to_csv(csv_images_path, index=False, encoding="utf-8-sig")

print(df_images.head())
print("✅ Images + mapping tables OK")

                  pdf_name  page_number                        img_filename  \
0  0-آثارنا المنهوبة_3.pdf            1  0-آثارنا المنهوبة_3_page1_img3.png   
1  0-آثارنا المنهوبة_3.pdf            1  0-آثارنا المنهوبة_3_page1_img4.png   
2  0-آثارنا المنهوبة_3.pdf            1  0-آثارنا المنهوبة_3_page1_img5.png   
3  0-آثارنا المنهوبة_3.pdf            2  0-آثارنا المنهوبة_3_page2_img3.png   
4  0-آثارنا المنهوبة_3.pdf            2  0-آثارنا المنهوبة_3_page2_img4.png   

                               chunk_id  table_idx  centre_x  centre_y  
0  0-آثارنا المنهوبة_3.pdf_page1_table2          2   140.300   371.795  
1  0-آثارنا المنهوبة_3.pdf_page1_table1          1   153.750   166.695  
2  0-آثارنا المنهوبة_3.pdf_page1_table3          3   141.025   614.855  
3  0-آثارنا المنهوبة_3.pdf_page2_table3          3   170.625   607.345  
4  0-آثارنا المنهوبة_3.pdf_page2_table1          1   191.525   139.745  
✅ Images + mapping tables OK
